# Week 22 Learning Notebook

This notebook mirrors the daily lesson plan and includes runnable demo code cells.

In [ ]:
from datetime import date
import warnings
import numpy as np
import pandas as pd

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from pandas_datareader import data as pdr
except Exception:
    pdr = None

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.6f}'.format

def _synthetic_price_frame(tickers, start='2018-01-01', periods=900, seed=7):
    idx = pd.bdate_range(start, periods=periods)
    out = pd.DataFrame(index=idx)
    rng = np.random.default_rng(seed)
    for i, ticker in enumerate(tickers):
        drift = 0.0002 + 0.00005 * (i + 1)
        vol = 0.010 + 0.002 * i
        rets = rng.normal(drift, vol, size=len(idx))
        out[ticker] = 100.0 * np.exp(np.cumsum(rets))
    return out

def load_market_prices(tickers, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    tickers = list(tickers)
    if yf is None:
        warnings.warn('yfinance unavailable, using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

    try:
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            if 'Close' in raw.columns.get_level_values(0):
                close = raw['Close']
            else:
                close = raw.xs('Close', axis=1, level=0, drop_level=True)
        else:
            close = raw.rename(columns={raw.columns[0]: tickers[0]})
        close = close.reindex(columns=tickers)
        close = close.dropna(how='all').ffill().dropna()
        if close.empty:
            raise ValueError('empty market data from Yahoo Finance')
        return close
    except Exception as exc:
        warnings.warn(f'Yahoo download failed ({exc}); using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

def load_fred_series(series_id, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    if pdr is None:
        warnings.warn('pandas_datareader unavailable, using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

    try:
        ser = pdr.DataReader(series_id, 'fred', start, end)[series_id]
        ser = ser.ffill().dropna()
        if ser.empty:
            raise ValueError('empty FRED series')
        return ser
    except Exception as exc:
        warnings.warn(f'FRED download failed ({exc}); using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

# Week 22 Day 01: Probability drill framework

## Study Duration
- Planned effort: 4 hours

## 4-Hour Lesson Flow
- 60 minutes: concept breakdown and formula derivation
- 75 minutes: real-market case study with data alignment checks
- 60 minutes: step-by-step quantitative problem solving
- 45 minutes: coding walkthrough and output verification

## Why It Matters in Quant
Build speed and accuracy for core quantitative interview domains.

## Continuity and Handoff
- Previous checkpoint: Week 21 Day 07: Portfolio Project
- Previous lesson file: content/week-21/day-07.md
- Today's deliverable: Create a probability drill notebook with timed question sets.
- Next handoff target: Week 22 Day 02: Statistics and estimation interview drills
- Next lesson file: content/week-22/day-02.md

## Theory Concepts

### Concept 1: Conditional probability
Conditional probability should be treated as a measurable component of 'Quant interview prep I'. For this week, emphasize decision quality, communication rigor, and reproducible evidence. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 2: Bayes intuition
Bayes intuition should be treated as a measurable component of 'Quant interview prep I'. For this week, emphasize decision quality, communication rigor, and reproducible evidence. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 3: Expected value shortcuts
Expected value shortcuts should be treated as a measurable component of 'Quant interview prep I'. For this week, emphasize decision quality, communication rigor, and reproducible evidence. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

## Mathematical Foundations (LaTeX)
### Formula 1: Expected Value
$$
EV=p\cdot Gain-(1-p)\cdot Loss
$$
Decision-quality baseline.

### Formula 2: Readiness Score
$$
S=\sum_j w_js_j
$$
Weighted progress metric.

### Formula 3: Bayes Update
$$
P(H\mid D)=\frac{P(D\mid H)P(H)}{P(D)}
$$
Evidence-driven belief update.

## Symbol Definitions
- $P_t$: price at time $t$
- $r_t$: simple return
- $\mu$: expected return
- $\sigma$: volatility
- $S$: readiness score
- $EV$: expected value
- $Gap_j$: target gap by skill

## Real Trading Example
- Instruments: SPY, QQQ, TLT
- Macro overlay (FRED): VIXCLS, TEDRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Solve coin/card-style probability questions under time constraints.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Probability drill framework'.

## Step-by-Step Solved Problems
### Solved Problem 1: Expected value
Given:
- Win probability=0.58, gain=1.5R, loss=1R.
Solution:
1. $EV=p\cdot Gain-(1-p)\cdot Loss$.
2. EV = 0.58*1.5 - 0.42*1.0 = 0.45R.
Final answer: Expected value = 0.45R per trade.

### Solved Problem 2: Readiness score
Given:
- Weights=[0.45,0.35,0.20], scores=[82,87,80].
Solution:
1. $S=\sum_jw_js_j$.
2. S = 0.45*82 + 0.35*87 + 0.20*80 = 83.35.
Final answer: Readiness score = 83.35/100.

### Solved Problem 3: CAGR target
Given:
- V0=1.00, VT=1.32, T=3 years.
Solution:
1. $CAGR=(V_T/V_0)^{1/T}-1$.
2. CAGR = (1.32/1.00)^(1/3)-1 = 0.0969.
Final answer: Required CAGR = 9.69%.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Create a probability drill notebook with timed question sets.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
readiness = score_readiness(quiz_scores, mock_scores, project_rubrics)
posterior = bayes_update(prior=0.55, likelihood=0.72, evidence=0.61)
roadmap = build_90_day_plan(readiness, posterior)
export_launch_checklist(roadmap)
```

## Practice Problems
1. Re-derive all formulas manually and explain each variable.
2. Re-run the real trading example using one alternate ticker.
3. Stress-test one assumption and write a risk-control rule.
4. Extend the code walkthrough with one new validation test.

## Reflection Question
Which probability pattern still feels slow or error-prone?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


In [ ]:
# Integration demo: readiness tracker anchored to market context
np.random.seed(2201)
spy = load_market_prices(['SPY'], start='2018-01-01')['SPY']
ret = spy.pct_change().dropna()
macro_unrate = load_fred_series('UNRATE', start='2018-01-01').dropna()

ann_return = float((1 + ret).prod() ** (252 / len(ret)) - 1)
ann_vol = float(ret.std() * np.sqrt(252))
sharpe = float((ann_return - 0.03) / max(ann_vol, 1e-8))

quiz_avg = float(np.clip(70 + 10 * sharpe, 60, 95))
mock_scores = np.clip(np.array([72, 78, 81]) + sharpe * 5, 60, 95)
macro_level = float(macro_unrate.iloc[-1]) if not macro_unrate.empty else float('nan')

{
    'spy_annualized_return': ann_return,
    'spy_annualized_vol': ann_vol,
    'spy_sharpe_proxy': sharpe,
    'latest_unemployment_rate': macro_level,
    'quiz_average': round(quiz_avg, 2),
    'mock_scores': [float(round(x, 2)) for x in mock_scores],
}


# Week 22 Day 02: Statistics and estimation interview drills

## Study Duration
- Planned effort: 4 hours

## 4-Hour Lesson Flow
- 60 minutes: concept breakdown and formula derivation
- 75 minutes: real-market case study with data alignment checks
- 60 minutes: step-by-step quantitative problem solving
- 45 minutes: coding walkthrough and output verification

## Why It Matters in Quant
Build speed and accuracy for core quantitative interview domains.

## Continuity and Handoff
- Previous checkpoint: Week 22 Day 01: Probability drill framework
- Previous lesson file: content/week-22/day-01.md
- Today's deliverable: Generate short stats-question answer templates.
- Next handoff target: Week 22 Day 03: Python coding speed drills
- Next lesson file: content/week-22/day-03.md

## Theory Concepts

### Concept 1: Estimator reasoning
Estimator reasoning should be treated as a measurable component of 'Quant interview prep I'. For this week, emphasize decision quality, communication rigor, and reproducible evidence. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 2: Bias-variance tradeoff
Bias-variance tradeoff should be treated as a measurable component of 'Quant interview prep I'. For this week, emphasize decision quality, communication rigor, and reproducible evidence. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 3: Hypothesis interpretation
Hypothesis interpretation should be treated as a measurable component of 'Quant interview prep I'. For this week, emphasize decision quality, communication rigor, and reproducible evidence. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

## Mathematical Foundations (LaTeX)
### Formula 1: Readiness Score
$$
S=\sum_j w_js_j
$$
Weighted progress metric.

### Formula 2: Bayes Update
$$
P(H\mid D)=\frac{P(D\mid H)P(H)}{P(D)}
$$
Evidence-driven belief update.

### Formula 3: CAGR
$$
CAGR=\left(\frac{V_T}{V_0}\right)^{1/T}-1
$$
Long-horizon growth target.

## Symbol Definitions
- $P_t$: price at time $t$
- $r_t$: simple return
- $\mu$: expected return
- $\sigma$: volatility
- $S$: readiness score
- $EV$: expected value
- $Gap_j$: target gap by skill

## Real Trading Example
- Instruments: SPY, QQQ, TLT
- Macro overlay (FRED): VIXCLS, TEDRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Explain confidence interval meaning in interview-friendly language.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Statistics and estimation interview drills'.

## Step-by-Step Solved Problems
### Solved Problem 1: Expected value
Given:
- Win probability=0.58, gain=1.5R, loss=1R.
Solution:
1. $EV=p\cdot Gain-(1-p)\cdot Loss$.
2. EV = 0.58*1.5 - 0.42*1.0 = 0.45R.
Final answer: Expected value = 0.45R per trade.

### Solved Problem 2: Readiness score
Given:
- Weights=[0.45,0.35,0.20], scores=[82,87,80].
Solution:
1. $S=\sum_jw_js_j$.
2. S = 0.45*82 + 0.35*87 + 0.20*80 = 83.35.
Final answer: Readiness score = 83.35/100.

### Solved Problem 3: CAGR target
Given:
- V0=1.00, VT=1.32, T=3 years.
Solution:
1. $CAGR=(V_T/V_0)^{1/T}-1$.
2. CAGR = (1.32/1.00)^(1/3)-1 = 0.0969.
Final answer: Required CAGR = 9.69%.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Generate short stats-question answer templates.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
readiness = score_readiness(quiz_scores, mock_scores, project_rubrics)
posterior = bayes_update(prior=0.55, likelihood=0.72, evidence=0.61)
roadmap = build_90_day_plan(readiness, posterior)
export_launch_checklist(roadmap)
```

## Practice Problems
1. Re-derive all formulas manually and explain each variable.
2. Re-run the real trading example using one alternate ticker.
3. Stress-test one assumption and write a risk-control rule.
4. Extend the code walkthrough with one new validation test.

## Reflection Question
How can you avoid over-talking while keeping precision?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


In [ ]:
# Integration demo: readiness tracker anchored to market context
np.random.seed(2202)
spy = load_market_prices(['SPY'], start='2018-01-01')['SPY']
ret = spy.pct_change().dropna()
macro_unrate = load_fred_series('UNRATE', start='2018-01-01').dropna()

ann_return = float((1 + ret).prod() ** (252 / len(ret)) - 1)
ann_vol = float(ret.std() * np.sqrt(252))
sharpe = float((ann_return - 0.03) / max(ann_vol, 1e-8))

quiz_avg = float(np.clip(70 + 10 * sharpe, 60, 95))
mock_scores = np.clip(np.array([72, 78, 81]) + sharpe * 5, 60, 95)
macro_level = float(macro_unrate.iloc[-1]) if not macro_unrate.empty else float('nan')

{
    'spy_annualized_return': ann_return,
    'spy_annualized_vol': ann_vol,
    'spy_sharpe_proxy': sharpe,
    'latest_unemployment_rate': macro_level,
    'quiz_average': round(quiz_avg, 2),
    'mock_scores': [float(round(x, 2)) for x in mock_scores],
}


# Week 22 Day 03: Python coding speed drills

## Study Duration
- Planned effort: 4 hours

## 4-Hour Lesson Flow
- 60 minutes: concept breakdown and formula derivation
- 75 minutes: real-market case study with data alignment checks
- 60 minutes: step-by-step quantitative problem solving
- 45 minutes: coding walkthrough and output verification

## Why It Matters in Quant
Build speed and accuracy for core quantitative interview domains.

## Continuity and Handoff
- Previous checkpoint: Week 22 Day 02: Statistics and estimation interview drills
- Previous lesson file: content/week-22/day-02.md
- Today's deliverable: Build a local timed coding drill harness.
- Next handoff target: Week 22 Day 04: SQL and data manipulation interview drills
- Next lesson file: content/week-22/day-04.md

## Theory Concepts

### Concept 1: Array and string patterns
Array and string patterns should be treated as a measurable component of 'Quant interview prep I'. For this week, emphasize decision quality, communication rigor, and reproducible evidence. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 2: Complexity awareness
Complexity awareness should be treated as a measurable component of 'Quant interview prep I'. For this week, emphasize decision quality, communication rigor, and reproducible evidence. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 3: Edge-case handling
Edge-case handling should be treated as a measurable component of 'Quant interview prep I'. For this week, emphasize decision quality, communication rigor, and reproducible evidence. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

## Mathematical Foundations (LaTeX)
### Formula 1: Bayes Update
$$
P(H\mid D)=\frac{P(D\mid H)P(H)}{P(D)}
$$
Evidence-driven belief update.

### Formula 2: CAGR
$$
CAGR=\left(\frac{V_T}{V_0}\right)^{1/T}-1
$$
Long-horizon growth target.

### Formula 3: Gap
$$
Gap_j=Target_j-Current_j
$$
Remaining improvement workload.

## Symbol Definitions
- $P_t$: price at time $t$
- $r_t$: simple return
- $\mu$: expected return
- $\sigma$: volatility
- $S$: readiness score
- $EV$: expected value
- $Gap_j$: target gap by skill

## Real Trading Example
- Instruments: SPY, QQQ, TLT
- Macro overlay (FRED): VIXCLS, TEDRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Solve 3 medium coding problems with clean test coverage.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Python coding speed drills'.

## Step-by-Step Solved Problems
### Solved Problem 1: Expected value
Given:
- Win probability=0.58, gain=1.5R, loss=1R.
Solution:
1. $EV=p\cdot Gain-(1-p)\cdot Loss$.
2. EV = 0.58*1.5 - 0.42*1.0 = 0.45R.
Final answer: Expected value = 0.45R per trade.

### Solved Problem 2: Readiness score
Given:
- Weights=[0.45,0.35,0.20], scores=[82,87,80].
Solution:
1. $S=\sum_jw_js_j$.
2. S = 0.45*82 + 0.35*87 + 0.20*80 = 83.35.
Final answer: Readiness score = 83.35/100.

### Solved Problem 3: CAGR target
Given:
- V0=1.00, VT=1.32, T=3 years.
Solution:
1. $CAGR=(V_T/V_0)^{1/T}-1$.
2. CAGR = (1.32/1.00)^(1/3)-1 = 0.0969.
Final answer: Required CAGR = 9.69%.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Build a local timed coding drill harness.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
readiness = score_readiness(quiz_scores, mock_scores, project_rubrics)
posterior = bayes_update(prior=0.55, likelihood=0.72, evidence=0.61)
roadmap = build_90_day_plan(readiness, posterior)
export_launch_checklist(roadmap)
```

## Practice Problems
1. Re-derive all formulas manually and explain each variable.
2. Re-run the real trading example using one alternate ticker.
3. Stress-test one assumption and write a risk-control rule.
4. Extend the code walkthrough with one new validation test.

## Reflection Question
Which coding pattern causes most debugging delays?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


In [ ]:
# Integration demo: readiness tracker anchored to market context
np.random.seed(2203)
spy = load_market_prices(['SPY'], start='2018-01-01')['SPY']
ret = spy.pct_change().dropna()
macro_unrate = load_fred_series('UNRATE', start='2018-01-01').dropna()

ann_return = float((1 + ret).prod() ** (252 / len(ret)) - 1)
ann_vol = float(ret.std() * np.sqrt(252))
sharpe = float((ann_return - 0.03) / max(ann_vol, 1e-8))

quiz_avg = float(np.clip(70 + 10 * sharpe, 60, 95))
mock_scores = np.clip(np.array([72, 78, 81]) + sharpe * 5, 60, 95)
macro_level = float(macro_unrate.iloc[-1]) if not macro_unrate.empty else float('nan')

{
    'spy_annualized_return': ann_return,
    'spy_annualized_vol': ann_vol,
    'spy_sharpe_proxy': sharpe,
    'latest_unemployment_rate': macro_level,
    'quiz_average': round(quiz_avg, 2),
    'mock_scores': [float(round(x, 2)) for x in mock_scores],
}


# Week 22 Day 04: SQL and data manipulation interview drills

## Study Duration
- Planned effort: 4 hours

## 4-Hour Lesson Flow
- 60 minutes: concept breakdown and formula derivation
- 75 minutes: real-market case study with data alignment checks
- 60 minutes: step-by-step quantitative problem solving
- 45 minutes: coding walkthrough and output verification

## Why It Matters in Quant
Build speed and accuracy for core quantitative interview domains.

## Continuity and Handoff
- Previous checkpoint: Week 22 Day 03: Python coding speed drills
- Previous lesson file: content/week-22/day-03.md
- Today's deliverable: Create SQL practice file with expected outputs.
- Next handoff target: Week 22 Day 05: Mock round synthesis
- Next lesson file: content/week-22/day-05.md

## Theory Concepts

### Concept 1: Joins and aggregations
Joins and aggregations should be treated as a measurable component of 'Quant interview prep I'. For this week, emphasize decision quality, communication rigor, and reproducible evidence. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 2: Window functions basics
Window functions basics should be treated as a measurable component of 'Quant interview prep I'. For this week, emphasize decision quality, communication rigor, and reproducible evidence. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 3: Data-quality checks
Data-quality checks should be treated as a measurable component of 'Quant interview prep I'. For this week, emphasize decision quality, communication rigor, and reproducible evidence. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

## Mathematical Foundations (LaTeX)
### Formula 1: CAGR
$$
CAGR=\left(\frac{V_T}{V_0}\right)^{1/T}-1
$$
Long-horizon growth target.

### Formula 2: Gap
$$
Gap_j=Target_j-Current_j
$$
Remaining improvement workload.

### Formula 3: Expected Value
$$
EV=p\cdot Gain-(1-p)\cdot Loss
$$
Decision-quality baseline.

## Symbol Definitions
- $P_t$: price at time $t$
- $r_t$: simple return
- $\mu$: expected return
- $\sigma$: volatility
- $S$: readiness score
- $EV$: expected value
- $Gap_j$: target gap by skill

## Real Trading Example
- Instruments: SPY, QQQ, TLT
- Macro overlay (FRED): VIXCLS, TEDRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Solve common SQL interview prompts on synthetic trade tables.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'SQL and data manipulation interview drills'.

## Step-by-Step Solved Problems
### Solved Problem 1: Expected value
Given:
- Win probability=0.58, gain=1.5R, loss=1R.
Solution:
1. $EV=p\cdot Gain-(1-p)\cdot Loss$.
2. EV = 0.58*1.5 - 0.42*1.0 = 0.45R.
Final answer: Expected value = 0.45R per trade.

### Solved Problem 2: Readiness score
Given:
- Weights=[0.45,0.35,0.20], scores=[82,87,80].
Solution:
1. $S=\sum_jw_js_j$.
2. S = 0.45*82 + 0.35*87 + 0.20*80 = 83.35.
Final answer: Readiness score = 83.35/100.

### Solved Problem 3: CAGR target
Given:
- V0=1.00, VT=1.32, T=3 years.
Solution:
1. $CAGR=(V_T/V_0)^{1/T}-1$.
2. CAGR = (1.32/1.00)^(1/3)-1 = 0.0969.
Final answer: Required CAGR = 9.69%.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Create SQL practice file with expected outputs.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
readiness = score_readiness(quiz_scores, mock_scores, project_rubrics)
posterior = bayes_update(prior=0.55, likelihood=0.72, evidence=0.61)
roadmap = build_90_day_plan(readiness, posterior)
export_launch_checklist(roadmap)
```

## Practice Problems
1. Re-derive all formulas manually and explain each variable.
2. Re-run the real trading example using one alternate ticker.
3. Stress-test one assumption and write a risk-control rule.
4. Extend the code walkthrough with one new validation test.

## Reflection Question
Which SQL operation do you mis-handle under pressure?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


In [ ]:
# Integration demo: readiness tracker anchored to market context
np.random.seed(2204)
spy = load_market_prices(['SPY'], start='2018-01-01')['SPY']
ret = spy.pct_change().dropna()
macro_unrate = load_fred_series('UNRATE', start='2018-01-01').dropna()

ann_return = float((1 + ret).prod() ** (252 / len(ret)) - 1)
ann_vol = float(ret.std() * np.sqrt(252))
sharpe = float((ann_return - 0.03) / max(ann_vol, 1e-8))

quiz_avg = float(np.clip(70 + 10 * sharpe, 60, 95))
mock_scores = np.clip(np.array([72, 78, 81]) + sharpe * 5, 60, 95)
macro_level = float(macro_unrate.iloc[-1]) if not macro_unrate.empty else float('nan')

{
    'spy_annualized_return': ann_return,
    'spy_annualized_vol': ann_vol,
    'spy_sharpe_proxy': sharpe,
    'latest_unemployment_rate': macro_level,
    'quiz_average': round(quiz_avg, 2),
    'mock_scores': [float(round(x, 2)) for x in mock_scores],
}


# Week 22 Day 05: Mock round synthesis

## Study Duration
- Planned effort: 4 hours

## 4-Hour Lesson Flow
- 60 minutes: concept breakdown and formula derivation
- 75 minutes: real-market case study with data alignment checks
- 60 minutes: step-by-step quantitative problem solving
- 45 minutes: coding walkthrough and output verification

## Why It Matters in Quant
Build speed and accuracy for core quantitative interview domains.

## Continuity and Handoff
- Previous checkpoint: Week 22 Day 04: SQL and data manipulation interview drills
- Previous lesson file: content/week-22/day-04.md
- Today's deliverable: Log mock performance and classify errors by type.
- Next handoff target: Week 22 Day 06: Revision Sprint
- Next lesson file: content/week-22/day-06.md

## Theory Concepts

### Concept 1: Time management
Time management should be treated as a measurable component of 'Quant interview prep I'. For this week, emphasize decision quality, communication rigor, and reproducible evidence. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 2: Answer structuring
Answer structuring should be treated as a measurable component of 'Quant interview prep I'. For this week, emphasize decision quality, communication rigor, and reproducible evidence. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 3: Error recovery strategy
Error recovery strategy should be treated as a measurable component of 'Quant interview prep I'. For this week, emphasize decision quality, communication rigor, and reproducible evidence. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

## Mathematical Foundations (LaTeX)
### Formula 1: Gap
$$
Gap_j=Target_j-Current_j
$$
Remaining improvement workload.

### Formula 2: Expected Value
$$
EV=p\cdot Gain-(1-p)\cdot Loss
$$
Decision-quality baseline.

### Formula 3: Readiness Score
$$
S=\sum_j w_js_j
$$
Weighted progress metric.

## Symbol Definitions
- $P_t$: price at time $t$
- $r_t$: simple return
- $\mu$: expected return
- $\sigma$: volatility
- $S$: readiness score
- $EV$: expected value
- $Gap_j$: target gap by skill

## Real Trading Example
- Instruments: SPY, QQQ, TLT
- Macro overlay (FRED): VIXCLS, TEDRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Run a mixed 45-minute mock round and self-score.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Mock round synthesis'.

## Step-by-Step Solved Problems
### Solved Problem 1: Expected value
Given:
- Win probability=0.58, gain=1.5R, loss=1R.
Solution:
1. $EV=p\cdot Gain-(1-p)\cdot Loss$.
2. EV = 0.58*1.5 - 0.42*1.0 = 0.45R.
Final answer: Expected value = 0.45R per trade.

### Solved Problem 2: Readiness score
Given:
- Weights=[0.45,0.35,0.20], scores=[82,87,80].
Solution:
1. $S=\sum_jw_js_j$.
2. S = 0.45*82 + 0.35*87 + 0.20*80 = 83.35.
Final answer: Readiness score = 83.35/100.

### Solved Problem 3: CAGR target
Given:
- V0=1.00, VT=1.32, T=3 years.
Solution:
1. $CAGR=(V_T/V_0)^{1/T}-1$.
2. CAGR = (1.32/1.00)^(1/3)-1 = 0.0969.
Final answer: Required CAGR = 9.69%.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Log mock performance and classify errors by type.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
readiness = score_readiness(quiz_scores, mock_scores, project_rubrics)
posterior = bayes_update(prior=0.55, likelihood=0.72, evidence=0.61)
roadmap = build_90_day_plan(readiness, posterior)
export_launch_checklist(roadmap)
```

## Practice Problems
1. Re-derive all formulas manually and explain each variable.
2. Re-run the real trading example using one alternate ticker.
3. Stress-test one assumption and write a risk-control rule.
4. Extend the code walkthrough with one new validation test.

## Reflection Question
What one behavior change would improve your next mock score most?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


In [ ]:
# Integration demo: readiness tracker anchored to market context
np.random.seed(2205)
spy = load_market_prices(['SPY'], start='2018-01-01')['SPY']
ret = spy.pct_change().dropna()
macro_unrate = load_fred_series('UNRATE', start='2018-01-01').dropna()

ann_return = float((1 + ret).prod() ** (252 / len(ret)) - 1)
ann_vol = float(ret.std() * np.sqrt(252))
sharpe = float((ann_return - 0.03) / max(ann_vol, 1e-8))

quiz_avg = float(np.clip(70 + 10 * sharpe, 60, 95))
mock_scores = np.clip(np.array([72, 78, 81]) + sharpe * 5, 60, 95)
macro_level = float(macro_unrate.iloc[-1]) if not macro_unrate.empty else float('nan')

{
    'spy_annualized_return': ann_return,
    'spy_annualized_vol': ann_vol,
    'spy_sharpe_proxy': sharpe,
    'latest_unemployment_rate': macro_level,
    'quiz_average': round(quiz_avg, 2),
    'mock_scores': [float(round(x, 2)) for x in mock_scores],
}


# Week 22 Day 06: Revision Sprint

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 22 Day 05: Mock round synthesis
- Previous lesson file: content/week-22/day-05.md
- Today's deliverable: Revision checklist with corrected errors and next-week retest priorities.
- Next handoff target: Week 22 Day 07: Portfolio Project
- Next lesson file: content/week-22/day-07.md

## Revision Plan
- 30 minutes: active recall of weekday concepts
- 40 minutes: rebuild one code workflow from memory
- 30 minutes: error log triage and retest plan
- 20 minutes: update progress tracker and notes

## Focus Areas
- Reattempt lowest-scoring question category
- Update interview error log
- Refine concise answer templates

## Revision Output
- [ ] Updated error log entries
- [ ] Weak concepts re-tested
- [ ] Next-week risk list prepared


# Week 22 Day 07: Portfolio Project

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 22 Day 06: Revision Sprint
- Previous lesson file: content/week-22/day-06.md
- Today's deliverable: Interview drill notebook pack
- Next handoff target: Week 23 Day 01: Market intuition drills
- Next lesson file: content/week-23/day-01.md

## Project Title
Interview drill notebook pack

## Problem Statement
Create a reusable set of timed quant interview drills with scoring logs.

## Data Sources
- Question bank
- Timed responses
- Error classifications

## Implementation Steps
1. Assemble mixed-domain question sets
2. Run timed attempts
3. Score and classify errors
4. Refine solution patterns
5. Build next-week focus plan

## Evaluation Metrics
- Accuracy
- Speed
- Error recurrence
- Communication clarity

## Deliverables
- Notebook or script output
- One-page summary memo
- Tracker update with completion and lessons learned
